# B0 / OSNet 256: подтверждение финального результата на трёх seed

Продолжение завершённого `variant_16/review_v1`. Первый final seed **20260915** уже готов и переиспользуется без обучения и пересчёта.
Добавляются только **20260916 и 20260917**, по две модели на seed: B0 и R1_resolution256.
Каждая обучается ровно **800 updates**, LR horizon остаётся **1700**. Итого **4 новых обучения / 3200 updates**.

Все четыре новые модели фиксируются до первой новой outer-оценки. Порог у каждой модели калибруется отдельно только на исходной calibration.
Результат — среднее и разброс по всем трём seed, **не выбор лучшего seed и не ансамбль**.
Решение продолжить принято после просмотра первого outer: это проверка устойчивости на development, не новый независимый тест.

Открой в прежнем `.venv`: **Restart Kernel → Run All**. Старый выполненный notebook не редактируй и не перезапускай одновременно.


In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/final_seed_confirmation.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из проекта Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
from IPython.display import Markdown, display
from training import osnet_review_protocol as review
from training.final_seed_confirmation import prepare, run

print('Repository:', REPO, '| Torch:', torch.__version__)


## 1. Настройки продолжения

Рецепты, seed, устройство и бюджеты наследуются из завершённого manifest, а не задаются заново.
`ALLOW_OUTER_EVALUATION=True` разрешает предусмотренную этой проверкой outer-оценку, но **не замену MVP**.
Не меняй бюджеты, исходники или runtime во время серии. NiVe, новые rerank-параметры и подбор порога по outer не добавляются.


In [ ]:
SOURCE_RUN = 'review_v1'
RUN_NAME = 'final_seeds_v1'
ALLOW_OUTER_EVALUATION = True
ALLOW_CPU_TRAINING = False

SOURCE_DIR = review.VARIANT / 'runs' / SOURCE_RUN
FROZEN = review.old.load_json(SOURCE_DIR / 'manifest.json')
DEVICE = FROZEN['runtime']['device']
torch.set_num_threads(FROZEN['runtime']['torch_threads'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(FROZEN['runtime']['deterministic'], warn_only=True)
print('Source:', SOURCE_RUN, '| new run:', RUN_NAME, '| device:', DEVICE)
print('Reused seed:', FROZEN['seeds'][0], '| additional seeds:', FROZEN['seeds'][1:])
print('4 new stages x 800 updates = 3200 updates. MVP unchanged.')


## 2. Проверка исходного финала — без обучения

Проверяются завершённые фазы, исходные веса/отчёты, CSV и изображения, frozen recipes, runtime и fingerprints.
Записывается только новый manifest в variant17. При несовпадении не удаляй старые manifests и не обходи проверку — нужен разбор причины.


In [ ]:
context = prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Output:', context['output'])
print('Fingerprint:', context['signature'])
print('Original final preserved:', context['manifest']['extension']['source_directory'])
print('Preflight passed. No optimizer updates yet.')


## 3. Четыре обучения и финальная оценка

Во время обучения выводятся модель, seed, шаг/800, затраченное время и ETA. Во время финальной оценки прогресс может временно не печататься.
После прерывания — **Restart Kernel → Run All** с теми же настройками. Завершённые модели и полностью завершённые оценки проверяются и переиспользуются.
Незавершённый training-блок до 200 updates повторится; если оценка/экспорт были прерваны, повторится эта оценка, а не обучение.

Исходный первый seed не копируется и не оценивается повторно. Его report включается в сводку с той же массой 1/3, что у каждого нового seed.


In [ ]:
result = run(context, allow_outer=ALLOW_OUTER_EVALUATION, allow_cpu=ALLOW_CPU_TRAINING)
print('Complete:', result['complete'], '| new updates:', result['new_optimizer_updates'])
print('Promoted:', result['promoted'], '| reused seed:', result['reused_seed'])


## 4. Сводный результат

Для разбора нужны `FINAL_RESULTS.md` и `results.json` в новом run-каталоге.
Отчёт показывает raw/reranked mAP@10, F1/TNR, каждый seed, среднее ± sample SD, paired R1−B0 и masked diagnostics.
Пороги не усредняются. Не выбираем лучший checkpoint/seed по outer, не меняем исходную validation и не продвигаем модель автоматически.
Исторический MVP показан отдельно: он не является четвёртым seed и не входит в средние.


In [ ]:
display(Markdown((context['output'] / 'FINAL_RESULTS.md').read_text(encoding='utf-8')))
print('Detailed results:', context['output'] / 'results.json')
print('Preserved first final:', SOURCE_DIR / 'final.json')
